# TalentAI Nexus - ML Model Training Notebook

This notebook demonstrates the complete ML lifecycle for candidate ranking:
1. Data Loading & Exploration
2. Data Cleaning
3. Feature Engineering
4. Model Training (Random Forest, XGBoost, Logistic Regression, Neural Network)
5. Hyperparameter Tuning
6. Evaluation
7. Model Deployment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import os

## 1. Data Loading

In [ ]:
df = pd.read_csv('../ml-service/data/sample_training_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Data Cleaning

In [ ]:
df = df.drop_duplicates()
df = df.fillna(df.median(numeric_only=True))
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates removed. Shape: {df.shape}')

## 3. Feature Engineering & Scaling

In [ ]:
FEATURES = ['skill_match_percentage', 'experience_score', 'education_relevance',
            'project_relevance', 'keyword_similarity', 'domain_matching']
TARGET = 'label'

X = df[FEATURES]
y = df[TARGET]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. Model Training & Hyperparameter Tuning

In [ ]:
models = {
    'Random Forest': (RandomForestClassifier(random_state=42), {
        'n_estimators': [50, 100, 200], 'max_depth': [5, 10, None]
    }),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=42), {
        'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1, 0.2]
    }),
    'Logistic Regression': (LogisticRegression(max_iter=1000), {
        'C': [0.1, 1, 10]
    }),
    'Neural Network': (MLPClassifier(max_iter=1000, random_state=42), {
        'hidden_layer_sizes': [(64, 32), (128, 64)], 'alpha': [0.001, 0.01]
    })
}

results = {}
for name, (model, params) in models.items():
    search = GridSearchCV(model, params, cv=3, scoring='f1')
    search.fit(X_train, y_train)
    y_pred = search.predict(X_test)
    results[name] = {
        'model': search.best_estimator_,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'best_params': search.best_params_
    }
    print(f'{name}: F1={results[name]["f1"]:.4f}, Acc={results[name]["accuracy"]:.4f}')

## 5. Evaluation & Confusion Matrix

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best_model = results[best_name]['model']
y_pred = best_model.predict(X_test)

print(f'Best Model: {best_name}')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 6. Save Model (.pkl)

In [ ]:
os.makedirs('../ml-service/models', exist_ok=True)
artifact = {'model': best_model, 'scaler': scaler, 'features': FEATURES}
joblib.dump(artifact, '../ml-service/models/notebook_trained_model.pkl')
print(f'Model saved: notebook_trained_model.pkl')